In [ ]:
import os, sys, gc, random, warnings
warnings.filterwarnings("ignore")
!pip install -q torch_geometric

if not os.path.exists("gnn-link-prediction"):
    !git clone -q https://github.com/dhanishta-r/gnn-link-prediction.git
if "gnn-link-prediction/src" not in sys.path:
    sys.path.append("gnn-link-prediction/src")

os.makedirs("results/m2", exist_ok=True)
OUT_CSV = "results/m2/raw_seal.csv"

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss, Conv1d, Linear, MaxPool1d
from scipy.sparse.csgraph import shortest_path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_sort_pool
from torch_geometric.utils import k_hop_subgraph, negative_sampling, to_scipy_sparse_matrix

from heart_data import load_heart
from evaluate import evaluate_model, append_results

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def drnl_node_labeling(adj, src, dst):
    adj_wo_dst = adj.copy()
    adj_wo_dst[dst, :] = 0
    adj_wo_dst[:, dst] = 0
    dist2src = shortest_path(adj_wo_dst, directed=False, unweighted=True, indices=src)

    adj_wo_src = adj.copy()
    adj_wo_src[src, :] = 0
    adj_wo_src[:, src] = 0
    dist2dst = shortest_path(adj_wo_src, directed=False, unweighted=True, indices=dst)

    dist = dist2src + dist2dst
    dist_over_2, dist_mod_2 = np.divmod(dist, 2)
    z = 1 + np.minimum(dist2src, dist2dst) + dist_over_2 * (dist_over_2 + dist_mod_2 - 1)
    z[src] = 1.
    z[dst] = 1.
    z[np.isnan(z)] = 0.
    z[np.isinf(z)] = 0.
    return torch.tensor(z, dtype=torch.long)

def extract_single_subgraph(edge_index, x, num_nodes, src, dst, y, num_hops):
    sub_nodes, sub_edge_index, mapping, _ = k_hop_subgraph(
        [src, dst], num_hops, edge_index, relabel_nodes=True, num_nodes=num_nodes
    )
    mask1 = (sub_edge_index[0] == mapping[0]) & (sub_edge_index[1] == mapping[1])
    mask2 = (sub_edge_index[0] == mapping[1]) & (sub_edge_index[1] == mapping[0])
    sub_edge_index = sub_edge_index[:, ~(mask1 | mask2)]

    adj = to_scipy_sparse_matrix(sub_edge_index, num_nodes=sub_nodes.size(0)).tocsr()
    z = drnl_node_labeling(adj, int(mapping[0]), int(mapping[1]))
    sub_x = x[sub_nodes] if x is not None else None
    return Data(x=sub_x, z=z, edge_index=sub_edge_index, y=torch.tensor([y], dtype=torch.float))

def extract_enclosing_subgraphs(edge_index, x, num_nodes, links, y, num_hops):
    links_np = links.cpu().numpy() if torch.is_tensor(links) else np.asarray(links)
    return [
        extract_single_subgraph(edge_index, x, num_nodes, int(links_np[i, 0]), int(links_np[i, 1]), y, num_hops)
        for i in range(links_np.shape[0])
    ]

class DGCNN(torch.nn.Module):
    def __init__(self, hidden_channels, num_layers, max_z, k=0.6, train_dataset=None, use_feature=True, dropout=0.5):
        super().__init__()
        self.use_feature = use_feature
        self.dropout = dropout
        if train_dataset is not None:
            num_nodes = sorted([d.num_nodes for d in train_dataset])
            k = num_nodes[int(np.ceil(k * len(num_nodes))) - 1]
            k = max(10, k)
        self.k = int(k)

        self.z_embedding = torch.nn.Embedding(max_z, hidden_channels)
        self.convs = torch.nn.ModuleList()
        in_channels = hidden_channels + (train_dataset[0].x.size(1) if use_feature else 0)
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, 1))

        total_latent_dim = hidden_channels * num_layers + 1
        conv1d_channels = [16, 32]
        conv1d_kws = [total_latent_dim, 5]
        self.conv1 = Conv1d(1, conv1d_channels[0], conv1d_kws[0], conv1d_kws[0])
        self.maxpool1d = MaxPool1d(2, 2)
        self.conv2 = Conv1d(conv1d_channels[0], conv1d_channels[1], conv1d_kws[1], 1)
        dense_dim = int((self.k - 2) / 2 + 1)
        dense_dim = (dense_dim - conv1d_kws[1] + 1) * conv1d_channels[1]
        self.lin1 = Linear(dense_dim, 128)
        self.lin2 = Linear(128, 1)

    def forward(self, z, edge_index, batch, x=None):
        z_emb = self.z_embedding(z.clamp(max=self.z_embedding.num_embeddings - 1))
        h = torch.cat([z_emb, x.to(torch.float)], dim=-1) if (self.use_feature and x is not None) else z_emb
        xs = [h]
        for conv in self.convs:
            xs.append(torch.tanh(conv(xs[-1], edge_index)))
        h = torch.cat(xs[1:], dim=-1)
        h = global_sort_pool(h, batch, self.k)
        h = h.unsqueeze(1)
        h = F.relu(self.conv1(h))
        h = self.maxpool1d(h)
        h = F.relu(self.conv2(h))
        h = h.view(h.size(0), -1)
        h = F.relu(self.lin1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.lin2(h).view(-1)

def train_and_build_scorer(data, x, edge_index, cfg, seed):
    set_seed(seed)
    num_nodes = x.size(0)
    pos_edges = torch.tensor(data.train_pos, dtype=torch.long)
    neg_edges = negative_sampling(edge_index, num_nodes=num_nodes, num_neg_samples=pos_edges.size(0)).t()

    pos_list = extract_enclosing_subgraphs(edge_index, x, num_nodes, pos_edges, 1, cfg["hops"])
    neg_list = extract_enclosing_subgraphs(edge_index, x, num_nodes, neg_edges, 0, cfg["hops"])
    train_dataset = pos_list + neg_list
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    model = DGCNN(
        hidden_channels=cfg["hidden"],
        num_layers=cfg["layers"],
        max_z=1000,
        train_dataset=train_dataset,
        dropout=cfg["dropout"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    criterion = BCEWithLogitsLoss()

    for _ in range(cfg["epochs"]):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.z, batch.edge_index, batch.batch, batch.x)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()

    del pos_list, neg_list, train_dataset, train_loader
    gc.collect()

    def score_fn(edges):
        model.eval()
        edges_np = edges.cpu().numpy() if torch.is_tensor(edges) else np.asarray(edges)
        total = edges_np.shape[0]
        chunk_size = 10000
        all_preds = []
        with torch.no_grad():
            for start in range(0, total, chunk_size):
                chunk_edges = edges_np[start:start + chunk_size]
                chunk_list = extract_enclosing_subgraphs(edge_index, x, num_nodes, chunk_edges, 0, cfg["hops"])
                chunk_loader = DataLoader(chunk_list, batch_size=1024, shuffle=False)
                for batch in chunk_loader:
                    batch = batch.to(device)
                    out = model(batch.z, batch.edge_index, batch.batch, batch.x)
                    all_preds.append(torch.sigmoid(out).cpu().numpy())
                del chunk_list, chunk_loader
        gc.collect()
        return np.concatenate(all_preds, axis=0)

    return score_fn, model

CONFIGS = [
    {"hops": 1, "hidden": 32, "layers": 3, "lr": 1e-3, "dropout": 0.5, "epochs": 15},
    {"hops": 1, "hidden": 32, "layers": 3, "lr": 5e-4, "dropout": 0.5, "epochs": 15},
    {"hops": 1, "hidden": 32, "layers": 3, "lr": 1e-3, "dropout": 0.2, "epochs": 15},
    {"hops": 1, "hidden": 64, "layers": 3, "lr": 1e-3, "dropout": 0.5, "epochs": 15},
    {"hops": 1, "hidden": 64, "layers": 3, "lr": 5e-4, "dropout": 0.5, "epochs": 15},
    {"hops": 1, "hidden": 64, "layers": 3, "lr": 1e-3, "dropout": 0.2, "epochs": 15},
]

for ds_name in ["cora", "citeseer", "pubmed"]:
    print(f"\n================ {ds_name.upper()} ================")
    data = load_heart(ds_name, root="gnn-link-prediction")
    x, edge_index = data.to_torch()

    best_cfg = CONFIGS[0]
    best_score = -1.0

    for cfg in CONFIGS:
        cfg_str = f"hops={cfg['hops']},hidden={cfg['hidden']},lr={cfg['lr']},drop={cfg['dropout']},ep={cfg['epochs']}"
        score_fn, model = train_and_build_scorer(data, x, edge_index, cfg, seed=0)
        rows = evaluate_model(score_fn, data, split="valid")
        append_results(OUT_CSV, rows, dataset=data.name, model="SEAL", config=cfg_str, seed=0, member="A", notes="tuning")
        heart_row = [r for r in rows if r["setting"] == "heart"][0]
        val_metric = heart_row["MRR"]
        print(f"[VALID] {cfg_str} -> HeaRT MRR: {val_metric:.2f}")
        if val_metric > best_score:
            best_score = val_metric
            best_cfg = cfg
        del score_fn, model
        torch.cuda.empty_cache()
        gc.collect()

    best_cfg_str = f"hops={best_cfg['hops']},hidden={best_cfg['hidden']},lr={best_cfg['lr']},drop={best_cfg['dropout']},ep={best_cfg['epochs']}"
    print(f"\n>>> BEST CONFIG FOR {ds_name}: {best_cfg_str} (Valid HeaRT MRR: {best_score:.2f}) <<<")
    for s in range(10):
        score_fn, model = train_and_build_scorer(data, x, edge_index, best_cfg, seed=s)
        val_rows = evaluate_model(score_fn, data, split="valid")
        test_rows = evaluate_model(score_fn, data, split="test")
        append_results(OUT_CSV, val_rows + test_rows, dataset=data.name, model="SEAL", config=best_cfg_str, seed=s, member="A", notes="final_10seeds")
        print(f"[FINAL SEED {s}] Done.")
        del score_fn, model
        torch.cuda.empty_cache()
        gc.collect()

print("\nALL EXPERIMENTS COMPLETED! results/m2/raw_seal.csv is ready.")